# 📦 NB1 — Données, Simulation du Bruit & Visualisations
**Projet 1 : Data-Centric Deep Learning for Noisy Labels**

Ce notebook couvre :
- Setup et reproductibilité
- Téléchargement et exploration de CIFAR-10
- Classe Clothing1M (bruit réel ~38%)
- Classe `NoisyDataset` : simulation du bruit symétrique et asymétrique
- Visualisation des matrices de transition et des exemples bruités
- Sauvegarde de la config partagée pour NB2/NB3/NB4

⏱️ **Durée estimée : ~15 minutes**

---
## 🔧 Section 0 — Setup & Reproductibilité

Fixer toutes les graines aléatoires est **impératif** pour la reproductibilité.
Sans ça, deux exécutions du même code peuvent produire des résultats
différents, ce qui invalide toute comparaison expérimentale.

In [ ]:
import random, os, pickle, warnings
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
os.environ['PYTHONHASHSEED']       = str(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.dpi']        = 120
plt.rcParams['font.size']         = 11
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print(f'✅ Seeds fixées à {SEED}')
print(f'💻 Device : {device}')
if torch.cuda.is_available():
    print(f'   GPU : {torch.cuda.get_device_name(0)}')

---
## 📦 Section 1 — CIFAR-10

**Pourquoi CIFAR-10 comme dataset principal ?**

CIFAR-10 est le benchmark standard dans la littérature sur le bruit de labels
(Han et al. 2018, Li et al. 2020) :
- Dataset **équilibré** (5000 images/classe) → pas de biais de classe.
- **Contrôle total** du bruit → on sait exactement quels labels sont corrompus.
- Paires de classes **visuellement proches** (cat/dog, deer/horse, automobile/truck)
  qui justifient le modèle de bruit asymétrique.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])

clean_trainset = torchvision.datasets.CIFAR10(
    './data', train=True,  download=True, transform=transform_train)
testset        = torchvision.datasets.CIFAR10(
    './data', train=False, download=True, transform=transform_test)

CLASSES     = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
N_CLASSES   = 10
NOISE_RATES = [0.0, 0.2, 0.4, 0.6]

counts = Counter(np.array(clean_trainset.targets))
print(f'✅ CIFAR-10 chargé')
print(f'   Train : {len(clean_trainset):,} | Test : {len(testset):,}')
print(f'   Exemples/classe : {counts[0]} (équilibré)')

In [ ]:
viz_set = torchvision.datasets.CIFAR10(
    './data', train=True, transform=transforms.ToTensor())
class_imgs = {i: [] for i in range(10)}
for img, lbl in viz_set:
    if len(class_imgs[lbl]) < 5:
        class_imgs[lbl].append(img)
    if all(len(v)==5 for v in class_imgs.values()): break

fig, axes = plt.subplots(10, 5, figsize=(9,18))
for ci in range(10):
    for col in range(5):
        ax = axes[ci][col]
        ax.imshow(class_imgs[ci][col].permute(1,2,0).numpy())
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASSES[ci], fontsize=10,
                          rotation=0, ha='right', va='center', labelpad=40)
plt.suptitle('CIFAR-10 — 10 classes × 5 exemples',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('cifar10_preview.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Les paires cat/dog, deer/horse, automobile/truck sont visuellement')
print('   proches → elles définissent le bruit ASYMÉTRIQUE utilisé dans NB2/NB3.')

---
## 🔀 Section 2 — Modélisation du Bruit

**Bruit symétrique** : erreurs aléatoires. Matrice de transition uniforme hors-diagonale.
$$T_{ij} = \begin{cases} 1-\varepsilon & i=j \\ \varepsilon/(K-1) & i\neq j \end{cases}$$

**Bruit asymétrique** : confusions cohérentes (Patrini et al. 2017).
Plus réaliste (Clothing1M), plus difficile à corriger.

In [ ]:
class NoisyDataset(Dataset):
    """
    Wrapper CIFAR-10 avec injection de bruit contrôlé.
    Retourne (image, noisy_label, index).
    L'index est nécessaire pour Reweighting et Small-Loss en NB3.
    """
    ASYM_MAP = {0:0, 1:2, 2:0, 3:5, 4:7,
                5:3, 6:6, 7:4, 8:8, 9:1}

    def __init__(self, dataset, noise_rate=0.0,
                 noise_type='symmetric', seed=42):
        assert 0.0 <= noise_rate < 1.0
        assert noise_type in ('symmetric','asymmetric')
        self.dataset      = dataset
        self.clean_labels = np.array(dataset.targets)
        self.noisy_labels = self.clean_labels.copy()
        if noise_rate > 0:
            rng = np.random.RandomState(seed)
            idx = rng.choice(len(self.clean_labels),
                             int(len(self.clean_labels)*noise_rate),
                             replace=False)
            for i in idx:
                orig = self.clean_labels[i]
                if noise_type == 'symmetric':
                    self.noisy_labels[i] = rng.choice(
                        [c for c in range(10) if c != orig])
                else:
                    t = self.ASYM_MAP[orig]
                    if t != orig: self.noisy_labels[i] = t
        self.noise_mask        = self.noisy_labels != self.clean_labels
        self.actual_noise_rate = self.noise_mask.mean()

    def get_transition_matrix(self):
        T = np.zeros((10,10))
        for c, n in zip(self.clean_labels, self.noisy_labels):
            T[c][n] += 1
        return T / T.sum(axis=1, keepdims=True)

    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        return img, self.noisy_labels[idx], idx


print('✅ NoisyDataset défini')
print()
print(f"{'Type':<13} {'Demandé':>9} {'Réel':>9}")
print('-'*34)
for nr in NOISE_RATES:
    s = NoisyDataset(clean_trainset, nr, 'symmetric')
    a = NoisyDataset(clean_trainset, nr, 'asymmetric')
    print(f"{'Symétrique':<13} {nr:>9.0%} {s.actual_noise_rate:>9.2%}")
    print(f"{'Asymétrique':<13} {nr:>9.0%} {a.actual_noise_rate:>9.2%}")
    print()
print('📊 Asymétrique : taux réel < taux demandé car airplane, frog, ship')
print('   n\'ont pas de confusion définie (T[i,i]=1.0 pour ces classes).')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,6))
for ax, (ntype, title) in zip(axes,[
    ('symmetric',  'Bruit Symétrique (ε=40%)'),
    ('asymmetric', 'Bruit Asymétrique (ε=40%)'),
]):
    T  = NoisyDataset(clean_trainset, 0.4, ntype).get_transition_matrix()
    im = ax.imshow(T, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(CLASSES, fontsize=9)
    ax.set_xlabel('Label observé (bruité)')
    ax.set_ylabel('Vrai label (propre)')
    ax.set_title(title, fontweight='bold', pad=12)
    for i in range(10):
        for j in range(10):
            c = 'white' if T[i,j] > 0.5 else 'black'
            ax.text(j,i,f'{T[i,j]:.2f}',ha='center',va='center',fontsize=7,color=c)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Matrices de Transition T[i,j] = P(label bruité=j | vrai label=i)',
             fontsize=13,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig('transition_matrices.png',dpi=150,bbox_inches='tight')
plt.show()
print('📊 Symétrique : diagonale réduite UNIFORMÉMENT.')
print('   Asymétrique : blocs hors-diagonale sur paires spécifiques.')
print('   Certaines lignes restent à 1.0 (classes non corrompues).')

In [ ]:
viz_base  = torchvision.datasets.CIFAR10(
    './data', train=True, transform=transforms.ToTensor())
noisy_viz = NoisyDataset(viz_base, 0.4, 'asymmetric')
noisy_idx = np.where(noisy_viz.noise_mask)[0]
selected  = np.random.RandomState(0).choice(noisy_idx, 16, replace=False)

fig, axes = plt.subplots(4,4, figsize=(10,10))
for pi, si in enumerate(selected):
    ax = axes[pi//4][pi%4]
    img,_,_ = noisy_viz[si]
    ax.imshow(img.permute(1,2,0).numpy())
    ax.axis('off')
    tl = CLASSES[noisy_viz.clean_labels[si]]
    nl = CLASSES[noisy_viz.noisy_labels[si]]
    ax.set_title(f'✓ {tl}\n✗ {nl}', fontsize=8, color='darkred',
                 bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.8))
plt.suptitle('Labels bruités — Asymétrique ε=40%\n✓ vrai label  |  ✗ label bruité',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('noisy_examples.png',dpi=150,bbox_inches='tight')
plt.show()
print('📊 Les erreurs ont un sens visuel (cat→dog, deer→horse).')
print('   C\'est ce qui rend le bruit asymétrique plus difficile à corriger')
print('   que le bruit symétrique : les méthodes basées sur la loss sont')
print('   moins efficaces car ces erreurs ressemblent à des cas difficiles.')

---
## 🧥 Section 3 — Clothing1M (Bruit Réel)

Contrairement au bruit *simulé* de CIFAR-10, Clothing1M contient du bruit
*réel* (~38%) issu du web scraping : labels extraits automatiquement des
titres HTML → confusions cohérentes (veste→manteau). C'est le scénario
industriel que nos méthodes doivent gérer.

In [ ]:
USE_CLOTHING1M = False
clothing_path  = None

try:
    import kagglehub
    clothing_path  = kagglehub.dataset_download('trolukovich/clothing1m-dataset')
    USE_CLOTHING1M = True
    print(f'✅ Clothing1M téléchargé : {clothing_path}')
except Exception as e:
    print(f'⚠️  Clothing1M non disponible : {e}')
    print('   Pour l\'utiliser :')
    print('   1. Télécharger depuis https://github.com/Cysu/noisy_label')
    print('   2. Uploader sur Kaggle comme dataset privé')
    print('   3. Ajouter comme Input dans ce notebook')


class Clothing1MDataset(Dataset):
    """
    Clothing1M — 14 classes, bruit réel ~38%.
    Structure : root_dir/{split}_labels.txt  +  root_dir/images/
    """
    CLASSES = ['T-shirt','Shirt','Knitwear','Chiffon','Sweater',
               'Hoodie','Windbreaker','Jacket','Down Coat','Suit',
               'Shawl','Dress','Vest','Underwear']

    def __init__(self, root_dir, split='noisy_train',
                 max_samples=50000, transform=None):
        self.transform = transform
        ann = os.path.join(root_dir, f'{split}_labels.txt')
        if not os.path.exists(ann):
            raise FileNotFoundError(f'Annotation non trouvée : {ann}')
        self.samples = []
        with open(ann) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                parts = line.strip().split()
                if len(parts) == 2:
                    p, lbl = parts[0], int(parts[1])
                    full   = os.path.join(root_dir, p)
                    if os.path.exists(full):
                        self.samples.append((full, lbl))
        print(f'✅ Clothing1M [{split}] : {len(self.samples):,} images')

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img = Image.open(self.samples[idx][0]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.samples[idx][1], idx


clothing_tf_train = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
clothing_tf_test = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

print('\n✅ Classe Clothing1MDataset définie')

if USE_CLOTHING1M:
    clothing_train = Clothing1MDataset(
        clothing_path, split='noisy_train',
        max_samples=50000, transform=clothing_tf_train)
    clothing_test = Clothing1MDataset(
        clothing_path, split='clean_test',
        transform=clothing_tf_test)
    clothing_train_loader = DataLoader(
        clothing_train, batch_size=32, shuffle=True, num_workers=2)
    clothing_test_loader = DataLoader(
        clothing_test, batch_size=64, shuffle=False, num_workers=2)

---
## 💾 Section 4 — Sauvegarde

In [ ]:
shared = {
    'SEED': SEED, 'CLASSES': CLASSES, 'N_CLASSES': N_CLASSES,
    'NOISE_RATES': NOISE_RATES, 'USE_CLOTHING1M': USE_CLOTHING1M,
    'clothing_path': clothing_path,
}
with open('shared_config.pkl','wb') as f:
    pickle.dump(shared, f)

print('✅ shared_config.pkl sauvegardé')
print(f'   CIFAR-10     : 50k train | 10k test | 10 classes équilibrées')
print(f'   Clothing1M   : disponible = {USE_CLOTHING1M}')
print(f'   Noise rates  : {NOISE_RATES}')
print('\n➡️  Lancer NB2_baseline.ipynb')